**Download sample sheet from GDC**


In [4]:
import requests
import pandas as pd
import os

In [5]:


# 1. The 'Address' of the GDC API
url = "https://api.gdc.cancer.gov/files"

# 2. The 'Filters' (Asking for only BRCA and RNA-Seq data)
filters = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}}
    ]
}

# 3. The 'Fields' (The specific info we want to see)
# 'cases.submitter_id' is the actual Patient ID (e.g. TCGA-A1-A0SQ)
fields = "file_id,file_name,cases.submitter_id,cases.samples.sample_type"

# 4. Put it all together and send the request
params = {
    "filters": filters,
    "fields": fields,
    "format": "JSON",
    "size": "2000"  # To make sure we get all ~1,200 BRCA patients
}

print("Searching for BRCA files...")
response = requests.post(url, json=params) # Using POST as in your example
data = response.json()["data"]["hits"]

# 5. Clean the data so a beginner can read it
clean_list = []
for item in data:
    clean_list.append({
        "File_ID": item["file_id"],
        "File_Name": item["file_name"],
        "Patient_ID": item["cases"][0]["submitter_id"], # The "same patient" link
        "Type": item["cases"][0]["samples"][0]["sample_type"]
    })

# 6. Save to a CSV file
df = pd.DataFrame(clean_list)
df.to_csv("my_brca_sample_sheet.csv", index=False)

print("Done! You now have 'my_brca_sample_sheet.csv' with all the links.")
print(df.head()) # Shows the first 5 rows on your screen

Searching for BRCA files...
Done! You now have 'my_brca_sample_sheet.csv' with all the links.
                                File_ID  \
0  744a6d3d-b666-49aa-8d26-47f34e3d1eb5   
1  4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5   
2  1ace2a0c-773d-45b5-8fd6-968c88731bbb   
3  2d5b0962-b5c4-4482-9f28-47e4dcdb6df6   
4  84cff855-d17a-4c09-b432-53893ae69c1e   

                                           File_Name    Patient_ID  \
0  94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...  TCGA-BH-A18H   
1  df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...  TCGA-E2-A14P   
2  c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...  TCGA-AN-A04A   
3  602e3d24-a45d-4fe4-9b70-a679d93f470d.rna_seq.a...  TCGA-5L-AAT1   
4  03d891b3-8faf-4384-94ce-2015f1ca5df0.rna_seq.a...  TCGA-GI-A2C8   

                  Type  
0        Primary Tumor  
1        Primary Tumor  
2        Primary Tumor  
3        Primary Tumor  
4  Solid Tissue Normal  


In [6]:
df

,File_ID,File_Name,Patient_ID,Type
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,TCGA-BH-A18H,Primary Tumor
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,TCGA-E2-A14P,Primary Tumor
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,TCGA-AN-A04A,Primary Tumor
3,2d5b0962-b5c4-4482-9f28-47e4dcdb6df6,602e3d24-a45d-4fe4-9b70-a679d93f470d.rna_seq.a...,TCGA-5L-AAT1,Primary Tumor
4,84cff855-d17a-4c09-b432-53893ae69c1e,03d891b3-8faf-4384-94ce-2015f1ca5df0.rna_seq.a...,TCGA-GI-A2C8,Solid Tissue Normal
...,...,...,...,...
1226,fae4edd8-db77-4937-94c4-3b16d39fb8c9,aa3cbc51-b74d-44b1-9043-cb2b9855a72b.rna_seq.a...,TCGA-GM-A2D9,Primary Tumor
1227,236805b9-ea89-43b4-ba51-c09276778e90,0bceb3fb-af69-4427-8ce1-767da4b31c37.rna_seq.a...,TCGA-LL-A5YP,Primary Tumor
1228,b9220098-f6f9-4732-8e94-3c9eeb4b8479,c84dbd50-ff69-4824-afe5-1eaec5780d27.rna_seq.a...,TCGA-E2-A1II,Primary Tumor
1229,f377c267-d5eb-4542-bd0d-e87f957f3154,eed349c6-a5ed-4543-ae8d-67b394b26c7a.rna_seq.a...,TCGA-D8-A1JE,Primary Tumor


In [7]:


# 1. YOUR FOLDER PATH
# Using the exact path you provided
save_path = 'C:/Users/abrup/Desktop/Project/survival-analysis/data/raw/gene_expression'

# Create the folder if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Successfully created: {save_path}")

# 2. LOAD YOUR LIST
# Make sure 'my_brca_sample_sheet.csv' is in the same folder as this script
manifest = pd.read_csv("my_brca_sample_sheet.csv")
all_ids = manifest['File_ID'].tolist()

print(f"Found {len(all_ids)} BRCA files to download.")

# 3. DOWNLOAD FROM GDC API
base_url = "https://api.gdc.cancer.gov/data/"

for index, fid in enumerate(all_ids):
    file_dest = os.path.join(save_path, f"{fid}.tsv")
    
    # Skip if already downloaded (Save time!)
    if os.path.exists(file_dest):
        continue

    # Progress tracker
    print(f"[{index + 1}/{len(all_ids)}] Downloading to Desktop...")

    try:
        response = requests.get(base_url + fid, timeout=30)
        if response.status_code == 200:
            with open(file_dest, "wb") as f:
                f.write(response.content)
        else:
            print(f"Error on {fid}: {response.status_code}")
            
    except Exception as e:
        print(f"Network error on {fid}: {e}")

print(f"\nSuccess! Your files are waiting for you at: {save_path}")

Found 1231 BRCA files to download.
[1/1231] Downloading to Desktop...
[2/1231] Downloading to Desktop...
[3/1231] Downloading to Desktop...
[4/1231] Downloading to Desktop...
[5/1231] Downloading to Desktop...
[6/1231] Downloading to Desktop...
[7/1231] Downloading to Desktop...
[8/1231] Downloading to Desktop...
[9/1231] Downloading to Desktop...
[10/1231] Downloading to Desktop...
[11/1231] Downloading to Desktop...
[12/1231] Downloading to Desktop...
[13/1231] Downloading to Desktop...
[14/1231] Downloading to Desktop...
[15/1231] Downloading to Desktop...
[16/1231] Downloading to Desktop...
[17/1231] Downloading to Desktop...
[18/1231] Downloading to Desktop...
[19/1231] Downloading to Desktop...
[20/1231] Downloading to Desktop...
[21/1231] Downloading to Desktop...
[22/1231] Downloading to Desktop...
[23/1231] Downloading to Desktop...
[24/1231] Downloading to Desktop...
[25/1231] Downloading to Desktop...
[26/1231] Downloading to Desktop...
[27/1231] Downloading to Desktop...
[2

KeyboardInterrupt: 

In [10]:
import requests
import pandas as pd
import os

# 1. Setup path
clinical_path = 'C:/Users/abrup/Desktop/Project/survival-analysis/data/raw/clinical'
if not os.path.exists(clinical_path):
    os.makedirs(clinical_path)

# 2. GDC API Endpoint for 'Cases' (where clinical info lives)
url = "https://api.gdc.cancer.gov/cases"

# 3. Ask for the specific 'Survival' fields we need
# 'submitter_id' matches your Sample Sheet Patient_ID
fields = [
    "submitter_id",
    "demographic.vital_status",
    "demographic.days_to_death",
    "demographic.days_to_last_follow_up",
    "demographic.gender",
    "demographic.age_at_index"
]

filters = {
    "op": "in",
    "content": {
        "field": "project.project_id",
        "value": ["TCGA-BRCA"]
    }
}

params = {
    "filters": filters,
    "fields": ",".join(fields),
    "format": "JSON",
    "size": "1500" # Covers all BRCA patients
}

print("Querying GDC for Clinical Survival data...")
response = requests.post(url, json=params)

if response.status_code == 200:
    data = response.json()["data"]["hits"]
    
    # 4. Flatten the data into a simple list
    clinical_list = []
    for item in data:
        demo = item.get("demographic", {})
        clinical_list.append({
            "Patient_ID": item["submitter_id"],
            "Vital_Status": demo.get("vital_status"),
            "Days_to_Death": demo.get("days_to_death"),
            "Days_to_FollowUp": demo.get("days_to_last_follow_up"),
            "Age": demo.get("age_at_index"),
            "Gender": demo.get("gender")
        })
    
    # 5. Save as a clean CSV
    df = pd.DataFrame(clinical_list)
    df.to_csv(os.path.join(clinical_path, "brca_clinical_simple.csv"), index=False)
    print(f"Success! Saved {len(df)} patient records to: {clinical_path}")
    print(df.head())
else:
    print(f"Failed! Error code: {response.status_code}")

Querying GDC for Clinical Survival data...
Success! Saved 1098 patient records to: C:/Users/abrup/Desktop/Project/survival-analysis/data/raw/clinical
     Patient_ID Vital_Status  Days_to_Death Days_to_FollowUp   Age  Gender
0  TCGA-E9-A5FL        Alive            NaN             None  65.0  female
1  TCGA-A2-A1G4        Alive            NaN             None  71.0  female
2  TCGA-BH-A0HF        Alive            NaN             None  77.0  female
3  TCGA-AR-A1AS        Alive            NaN             None  54.0  female
4  TCGA-D8-A13Y        Alive            NaN             None  52.0  female
